<p style="float: left;"><a href="pattern-matching.ipynb" target="_blank">Previous</a></p>
<p style="float: left;"><a href="_index.ipynb" target="_blank">Next</a></p>
<p style="text-align:center;">Tour of Scala</p>
<div style="clear: both;"></div>

# Exercises

Use Scala worksheet and the provided template repository to solve the following problems.

## This is about Set

Using the defintions of `trait Ordered[A]`, `class Set[T <: Ordered[T]]` and `case class Num(value: Double)`
provided in the [restricting generacity](upper-type-bounds.ipynb) notebook, implement the following methods defintion.

1) Write method `def union(...)` which performs the union of two sets.
2) Write method `def intersection(...)` which performs the intersection of two sets.
3) Add a method `def excl(x: Num)` to return the given set without the element `x`.

_Make sure to solve the problem the most functional way possible_.

In [ ]:
// code goes here...

In [5]:
import scala.language.postfixOps

trait Ordered[A] {
    def compare(that: A): Int
    def < (that: A): Boolean = (this compare that) < 0
    def > (that: A): Boolean = (this compare that) > 0
    def <= (that: A): Boolean = (this compare that) <= 0
    def >= (that: A): Boolean = (this compare that) >= 0
    def compareTo(that: A): Int = compare(that)
}

abstract class Set[T <: Ordered[T]] {
    def elem: T
    def left: Set[T]
    def right: Set[T]
    
    def incl(x: T): Set[T]
    def union(s: Set[T]): Set[T]
    def intersection(s: Set[T]): Set[T]
    def excl(x: T): Set[T]
    def contains(x: T): Boolean
    
    def inOrden: Unit = this match {
        case _: EmptySet[T] => print("")
        case that: NonEmptySet[T] => {
            left.inOrden
            print(that.elem.toString())
            right.inOrden
        }
    }

}

class EmptySet[T <: Ordered[T]] extends Set[T] {
    def elem = throw new Exception("EmptySet.elem not supported")
    def left = throw new Exception("EmptySet.left not supported")
    def right = throw new Exception("EmptySet.right not supported")
    
    def contains(x: T): Boolean = false
    
    def incl(x: T): Set[T] = new NonEmptySet(x, new EmptySet[T], new EmptySet[T])
    
    def union(s: Set[T]) = s

    def intersection(s: Set[T]) = new EmptySet[T]

    def excl(x: T) = new EmptySet[T]
}

class NonEmptySet[T <: Ordered[T]](e: T, lset: Set[T], rset: Set[T]) extends Set[T] {
    def elem = e
    def left = lset
    def right = rset

    def isEmpty = false
    
    def contains(x: T): Boolean =
        if (x < elem) left contains x // T needs to be comparable
        else if (x > elem) right contains x
        else true
    
    def incl(x: T): Set[T] =
        if (x < elem) new NonEmptySet(elem, left incl x, right)
        else if (x > elem) new NonEmptySet(elem, left, right incl x)
        else this
    
    def union(s: Set[T]) = ((left union right) union s) incl elem
    
    def intersection(s: Set[T]) =  {
        val leftIntersection = left intersection s
        val rightIntersection = right intersection s
        if (s contains elem) (leftIntersection union rightIntersection) incl elem
        else leftIntersection union rightIntersection    
    }

    def excl(x: T) = {
        if (elem == x) left union right
        else (left excl x) union (right excl x) incl elem
    }
}

case class Num(value: Double) extends Ordered[Num] {
    def compare(that: Num): Int =
        if (this.value < that.value) -1
        else if (this.value > that.value) 1
        else 0
}

import scala.language.postfixOps
defined trait Ordered
defined class Set
defined class EmptySet
defined class NonEmptySet
defined class Num

Union example

In [49]:
val s = new EmptySet[Num].incl(Num(9)).incl(Num(0)).incl(Num(10))
val r = new EmptySet[Num].incl(Num(4))
val union = r.union(s)
union.inOrden

Num(0.0)Num(4.0)Num(9.0)Num(10.0)

s: Set[Num] = ammonite.$sess.cmd48$Helper$NonEmptySet@4dec7192
r: Set[Num] = ammonite.$sess.cmd48$Helper$NonEmptySet@22a3f5fb
union: Set[Num] = ammonite.$sess.cmd48$Helper$NonEmptySet@27101994

Intersection example

In [52]:
val s = new EmptySet[Num].incl(Num(9)).incl(Num(0)).incl(Num(10)).incl(Num(4))
val r = new EmptySet[Num].incl(Num(4))
val intersection = r.intersection(s)
intersection.inOrden

Num(4.0)

s: Set[Num] = ammonite.$sess.cmd48$Helper$NonEmptySet@39eaf627
r: Set[Num] = ammonite.$sess.cmd48$Helper$NonEmptySet@8e6ed54
intersection: Set[Num] = ammonite.$sess.cmd48$Helper$NonEmptySet@45f82fbb

Exclude example

In [6]:
val s = new EmptySet[Num].incl(Num(9)).incl(Num(0)).incl(Num(10)).incl(Num(4))
val r = new EmptySet[Num].incl(Num(4)).incl(Num(10))
val intersection = r.intersection(s).excl(Num(4)).excl(Num(7))
intersection.inOrden

Num(10.0)

s: Set[Num] = ammonite.$sess.cmd5$Helper$NonEmptySet@51378faf
r: Set[Num] = ammonite.$sess.cmd5$Helper$NonEmptySet@5747ad67
intersection: Set[Num] = ammonite.$sess.cmd5$Helper$NonEmptySet@62c3e2e3

## Pattern matching!

Define the trait `Expr` and the case class `Operator`. Use
the case class `Num` to define the function `printExpr`.

This should be the behaviour of the function `printExpr`:

```scala
printExpr(Num(8))
It is a number: 8.0

printExpr(Operator("+"))
It is an operator: +
```

In [1]:
trait Expr

case class Num(value: Double) extends Ordered[Num] with Expr {
    def compare(that: Num): Int =
        if (this.value < that.value) -1
        else if (this.value > that.value) 1
        else 0
}

case class Operator(op: String) extends Expr

def printExpr(exp: Expr): Unit = exp match {
    case Num(num) => println(s"It is a number: ${num}")
    case Operator(op) => println(s"It is an operator: ${op}")
}

defined trait Expr
defined class Num
defined class Operator
defined function printExpr

In [2]:
printExpr(Num(8))

It is a number: 8.0


In [3]:
printExpr(Operator("+"))

It is an operator: +


<p style="float: left;"><a href="pattern-matching.ipynb" target="_blank">Previous</a></p>
<p style="float: left;"><a href="_index.ipynb" target="_blank">Next</a></p>
<p style="text-align:center;">Tour of Scala</p>
<div style="clear: both;"></div>